### Round 5 Analysis — Vertical Sleeping Pods

In [ ]:
import os
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.stats as stats

from pathlib import Path
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.stattools import jarque_bera

import prosperity4
from prosperity4.utils.statistics_utils import compute_returns
from prosperity4.utils.dataloader import (
    load_trading_data,
    get_product_data,
    get_day_data,
    get_product_day_data,
    get_price_data,
    get_order_book_data,
    get_volume_data,
    convert_timestamp,
)


plt.style.use("dark_background")
sns.set_palette("pastel")

### Data Loading

In [ ]:
REPO_ROOT = Path(prosperity4.__file__).parents[1]
DATA_FOLDER = REPO_ROOT / "prosperity4" / "round5" / "data"
ROUND_NUM = 5
DAYS = [2, 3, 4]
SYMBOLS = ['SLEEP_POD_SUEDE', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_POLYESTER', 'SLEEP_POD_NYLON', 'SLEEP_POD_COTTON']

data = load_trading_data(DATA_FOLDER, ROUND_NUM, DAYS)
prices_df = data.get("prices")
trades_df = data.get("trades")

print("Prices Shape :", prices_df.shape if prices_df is not None else None)
print("Trades Shape :", trades_df.shape if trades_df is not None else None)
print("\n--- Prices Head ---")
display(prices_df.head())
print("\n--- Trades Head ---")
display(trades_df.head())

### Helper Functions

In [ ]:
def compute_max_vol_fv(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    bid_prices  = df[["bid_price_1",  "bid_price_2",  "bid_price_3"]].values
    bid_volumes = df[["bid_volume_1", "bid_volume_2", "bid_volume_3"]].values
    ask_prices  = df[["ask_price_1",  "ask_price_2",  "ask_price_3"]].values
    ask_volumes = df[["ask_volume_1", "ask_volume_2", "ask_volume_3"]].values

    n = len(df)

    bid_all_nan = np.all(np.isnan(bid_volumes), axis=1)
    ask_all_nan = np.all(np.isnan(ask_volumes), axis=1)

    bid_argmax = np.where(bid_all_nan, 0, np.nanargmax(np.where(np.isnan(bid_volumes), -np.inf, bid_volumes), axis=1))
    ask_argmax = np.where(ask_all_nan, 0, np.nanargmax(np.where(np.isnan(ask_volumes), -np.inf, ask_volumes), axis=1))

    max_vol_bid = bid_prices[np.arange(n), bid_argmax]
    max_vol_ask = ask_prices[np.arange(n), ask_argmax]

    max_vol_bid = np.where(bid_all_nan, np.nan, max_vol_bid)
    max_vol_ask = np.where(ask_all_nan, np.nan, max_vol_ask)

    df["max_vol_bid"] = max_vol_bid
    df["max_vol_ask"] = max_vol_ask

    both_present = ~np.isnan(max_vol_bid) & ~np.isnan(max_vol_ask)
    only_bid     = ~np.isnan(max_vol_bid) &  np.isnan(max_vol_ask)
    only_ask     =  np.isnan(max_vol_bid) & ~np.isnan(max_vol_ask)

    raw_fv = np.where(both_present, (max_vol_bid + max_vol_ask) / 2,
             np.where(only_bid,     max_vol_bid,
             np.where(only_ask,     max_vol_ask, np.nan)))

    df["fv"] = pd.Series(raw_fv).ffill().values
    return df

# Price & Order Book Plots

In [ ]:
for SYMBOL in SYMBOLS:
    print(f"\n{'='*60}")
    print(f"Price & Order Book — {SYMBOL}")
    print(f"{'='*60}")

    product_df_raw = prices_df[prices_df["product"] == SYMBOL]
    product_trades_df = trades_df[trades_df["symbol"] == SYMBOL]

    product_trades = product_trades_df.copy()
    product_trades = product_trades.drop(columns=["currency"])
    product_trades = product_trades.groupby(
        ["timestamp", "price", "day", "buyer", "seller"], as_index=False
    ).agg({"quantity": "sum"})
    product_trades = product_trades.rename(columns={
        "price":    "market order price",
        "quantity": "market order quantity",
    })
    product_df = product_df_raw.merge(
        product_trades[["timestamp", "market order price", "market order quantity", "day", "buyer", "seller"]],
        on=["timestamp", "day"],
        how="left",
    )
    product_df = convert_timestamp(product_df)

    # Full price series
    plt.figure(figsize=(15, 5))
    plt.title(f"Price & Order Book — {SYMBOL}")
    plt.plot(product_df["t"], product_df["mid_price"],   color="white",  label="Mid Price")
    plt.plot(product_df["t"], product_df["ask_price_1"], color="red",    label="Ask 1")
    plt.plot(product_df["t"], product_df["ask_price_2"], color="salmon",  label="Ask 2", alpha=0.6)
    plt.plot(product_df["t"], product_df["bid_price_1"], color="lime",   label="Bid 1")
    plt.plot(product_df["t"], product_df["bid_price_2"], color="green",  label="Bid 2", alpha=0.6)
    plt.grid(True)
    plt.legend()
    plt.show()

    # Last 1000 ticks
    plt.figure(figsize=(15, 5))
    plt.title(f"Price & Order Book (last 1000 ticks) — {SYMBOL}")
    plt.plot(product_df["t"][-1000:], product_df["mid_price"][-1000:],   color="white",     alpha=0.5, label="Mid Price")
    plt.plot(product_df["t"][-1000:], product_df["ask_price_1"][-1000:], color="red",       alpha=1.0, label="Ask 1")
    plt.plot(product_df["t"][-1000:], product_df["ask_price_2"][-1000:], color="orange",    alpha=0.5, label="Ask 2")
    plt.plot(product_df["t"][-1000:], product_df["ask_price_3"][-1000:], color="salmon",    alpha=0.5, label="Ask 3")
    plt.plot(product_df["t"][-1000:], product_df["bid_price_1"][-1000:], color="lime",      alpha=1.0, label="Bid 1")
    plt.plot(product_df["t"][-1000:], product_df["bid_price_2"][-1000:], color="green",     alpha=0.5, label="Bid 2")
    plt.plot(product_df["t"][-1000:], product_df["bid_price_3"][-1000:], color="darkgreen", alpha=0.5, label="Bid 3")
    plt.legend()
    plt.show()

### FV + SMA + Z-Score

In [ ]:
SMA_WINDOW = 50

for SYMBOL in SYMBOLS:
    print(f"\n{'='*60}")
    print(f"FV + SMA + Z-Score — {SYMBOL}")
    print(f"{'='*60}")

    product_df_raw = prices_df[prices_df["product"] == SYMBOL]
    product_trades_df = trades_df[trades_df["symbol"] == SYMBOL]

    product_trades = product_trades_df.copy()
    product_trades = product_trades.drop(columns=["currency"])
    product_trades = product_trades.groupby(
        ["timestamp", "price", "day", "buyer", "seller"], as_index=False
    ).agg({"quantity": "sum"})
    product_trades = product_trades.rename(columns={
        "price":    "market order price",
        "quantity": "market order quantity",
    })
    product_df = product_df_raw.merge(
        product_trades[["timestamp", "market order price", "market order quantity", "day", "buyer", "seller"]],
        on=["timestamp", "day"],
        how="left",
    )
    product_df = convert_timestamp(product_df)
    product_df = compute_max_vol_fv(product_df)

    rolling_mean = product_df["fv"].rolling(window=SMA_WINDOW).mean()
    rolling_std  = product_df["fv"].rolling(window=SMA_WINDOW).std()
    product_df["zscore"] = (product_df["fv"] - rolling_mean) / rolling_std

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True,
                                    gridspec_kw={"height_ratios": [2, 1]})

    ax1.plot(product_df["t"], product_df["fv"],          color="white",  linewidth=1,   label="FV")
    ax1.plot(product_df["t"], product_df["bid_price_1"], color="lime",   linewidth=1,   label="Best Bid")
    ax1.plot(product_df["t"], product_df["ask_price_1"], color="red",    linewidth=1,   label="Best Ask")
    ax1.plot(product_df["t"], rolling_mean,              color="orange", linewidth=1,   label=f"SMA {SMA_WINDOW}")
    ax1.legend()
    ax1.grid(True)
    ax1.set_ylabel("Price")
    ax1.set_title(f"{SYMBOL} — Fair Value & SMA")

    ax2.plot(product_df["t"], product_df["zscore"], color="white", linewidth=0.8)
    ax2.axhline( 2, color="red",  linestyle="--", alpha=0.7, label="Z = +2")
    ax2.axhline(-2, color="lime", linestyle="--", alpha=0.7, label="Z = -2")
    ax2.axhline( 0, color="gray", linestyle=":",  alpha=0.5)
    ax2.fill_between(product_df["t"],  2, product_df["zscore"].clip(lower= 2), color="red",  alpha=0.15)
    ax2.fill_between(product_df["t"], -2, product_df["zscore"].clip(upper=-2), color="lime", alpha=0.15)
    ax2.set_ylabel("Z-Score")
    ax2.set_xlabel("Time")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

    # Rolling volatility
    VOL_WINDOW = 100
    product_df["rolling_vol"] = product_df["fv"].diff().rolling(window=VOL_WINDOW).std()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True,
                                    gridspec_kw={"height_ratios": [2, 1]})

    ax1.plot(product_df["t"], product_df["fv"], color="white", linewidth=0.8, label="FV (mid)")
    ax1.set_ylabel("Price")
    ax1.set_title(f"{SYMBOL} — Price & Rolling Volatility (window={VOL_WINDOW})")
    ax1.legend()
    ax1.grid(True)

    ax2.plot(product_df["t"], product_df["rolling_vol"], color="orange", linewidth=0.8,
             label=f"Volatility (σ, {VOL_WINDOW}-tick)")
    ax2.fill_between(product_df["t"], 0, product_df["rolling_vol"], color="orange", alpha=0.2)
    ax2.set_ylabel("Volatility (σ)")
    ax2.set_xlabel("Time")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

    print(f"Mean vol : {product_df['rolling_vol'].mean():.4f}")
    print(f"Max vol  : {product_df['rolling_vol'].max():.4f}")
    print(f"Std vol  : {product_df['rolling_vol'].std():.4f}")

---
## Returns Analysis

In [ ]:
for SYMBOL in SYMBOLS:
    print(f"\n{'='*60}")
    print(f"Returns Analysis — {SYMBOL}")
    print(f"{'='*60}")

    product_df_raw = prices_df[prices_df["product"] == SYMBOL]
    product_trades_df = trades_df[trades_df["symbol"] == SYMBOL]

    product_trades = product_trades_df.copy()
    product_trades = product_trades.drop(columns=["currency"])
    product_trades = product_trades.groupby(
        ["timestamp", "price", "day", "buyer", "seller"], as_index=False
    ).agg({"quantity": "sum"})
    product_trades = product_trades.rename(columns={
        "price":    "market order price",
        "quantity": "market order quantity",
    })
    product_df = product_df_raw.merge(
        product_trades[["timestamp", "market order price", "market order quantity", "day", "buyer", "seller"]],
        on=["timestamp", "day"],
        how="left",
    )
    product_df = convert_timestamp(product_df)
    product_df = compute_max_vol_fv(product_df)

    product_df["fv_return"] = product_df["fv"].diff()
    returns = product_df["fv_return"].dropna()

    # Histogram + fitted distributions
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))

    ax = axes[0]
    ax.hist(returns, bins=100, density=True, color="steelblue", alpha=0.7, label="Empirical")
    x = np.linspace(returns.min(), returns.max(), 500)
    mu, sigma = stats.norm.fit(returns)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), color="white", linewidth=1.5,
            label=f"Normal (μ={mu:.3f}, σ={sigma:.3f})")
    df_t, loc_t, scale_t = stats.t.fit(returns)
    ax.plot(x, stats.t.pdf(x, df_t, loc_t, scale_t), color="orange", linewidth=1.5,
            label=f"Student-t (df={df_t:.2f})")
    ax.set_title(f"FV Return Distribution — {SYMBOL}")
    ax.set_xlabel("Δ FV")
    ax.legend()
    ax.grid(True)

    # QQ plot
    ax = axes[1]
    (osm, osr), (slope, intercept, _) = stats.probplot(returns, dist="norm")
    ax.scatter(osm, osr, s=2, color="cyan", alpha=0.5)
    ax.plot(osm, slope * np.array(osm) + intercept, color="white", linewidth=1.5)
    ax.set_title(f"QQ Plot — Returns vs Normal — {SYMBOL}")
    ax.set_xlabel("Theoretical Quantiles")
    ax.set_ylabel("Sample Quantiles")
    ax.grid(True)

    plt.tight_layout()
    plt.show()

    print(f"Mean   : {returns.mean():.4f}")
    print(f"Std    : {returns.std():.4f}")
    print(f"Skew   : {returns.skew():.4f}")
    print(f"Kurt   : {returns.kurtosis():.4f}  (excess)")

    jb_stat, jb_p, skew_jb, kurt_jb = jarque_bera(returns)
    print(f"Jarque-Bera stat : {jb_stat:.2f}  p-value : {jb_p:.4f}")
    print("→ Returns are " + ("NOT " if jb_p < 0.05 else "") + "normally distributed at 5% level")

    # ACF / PACF
    fig, axes = plt.subplots(2, 2, figsize=(15, 7))
    plot_acf (returns,       ax=axes[0, 0], lags=50, color="cyan",   title=f"ACF — Returns — {SYMBOL}")
    plot_pacf(returns,       ax=axes[0, 1], lags=50, color="cyan",   title=f"PACF — Returns — {SYMBOL}", method="ywm")
    plot_acf (returns ** 2,  ax=axes[1, 0], lags=50, color="orange", title=f"ACF — Squared Returns — {SYMBOL}")
    plot_pacf(returns ** 2,  ax=axes[1, 1], lags=50, color="orange", title=f"PACF — Squared Returns — {SYMBOL}", method="ywm")
    for ax in axes.flat:
        ax.grid(True)
    plt.tight_layout()
    plt.show()

    # ADF stationarity test
    for series_name, series in [(f"FV ({SYMBOL})", product_df["fv"]), (f"FV Returns ({SYMBOL})", returns)]:
        result = adfuller(series.dropna())
        print(f"--- {series_name} ---")
        print(f"  ADF stat : {result[0]:.4f}")
        print(f"  p-value  : {result[1]:.4f}")
        print(f"  {'Stationary ✓' if result[1] < 0.05 else 'Non-stationary ✗'} at 5% level")
    print()

---
## Spread & Volume Imbalance Analysis

In [ ]:
for SYMBOL in SYMBOLS:
    print(f"\n{'='*60}")
    print(f"Spread & Volume Imbalance — {SYMBOL}")
    print(f"{'='*60}")

    product_df_raw = prices_df[prices_df["product"] == SYMBOL]
    product_trades_df = trades_df[trades_df["symbol"] == SYMBOL]

    product_trades = product_trades_df.copy()
    product_trades = product_trades.drop(columns=["currency"])
    product_trades = product_trades.groupby(
        ["timestamp", "price", "day", "buyer", "seller"], as_index=False
    ).agg({"quantity": "sum"})
    product_trades = product_trades.rename(columns={
        "price":    "market order price",
        "quantity": "market order quantity",
    })
    product_df = product_df_raw.merge(
        product_trades[["timestamp", "market order price", "market order quantity", "day", "buyer", "seller"]],
        on=["timestamp", "day"],
        how="left",
    )
    product_df = convert_timestamp(product_df)
    product_df = compute_max_vol_fv(product_df)

    product_df["spread"] = product_df["ask_price_1"] - product_df["bid_price_1"]

    total_bid_vol = product_df[["bid_volume_1", "bid_volume_2", "bid_volume_3"]].sum(axis=1)
    total_ask_vol = product_df[["ask_volume_1", "ask_volume_2", "ask_volume_3"]].sum(axis=1)
    product_df["vol_imbalance"] = (total_bid_vol - total_ask_vol) / (total_bid_vol + total_ask_vol)

    fig, axes = plt.subplots(2, 2, figsize=(15, 8))

    axes[0, 0].hist(product_df["spread"].dropna(), bins=50, color="cyan", alpha=0.8, density=True)
    axes[0, 0].set_title(f"Spread Distribution — {SYMBOL}")
    axes[0, 0].set_xlabel("Spread (ticks)")
    axes[0, 0].grid(True)

    axes[0, 1].plot(product_df["t"], product_df["spread"], color="cyan", linewidth=0.5, alpha=0.7)
    axes[0, 1].set_title(f"Spread Over Time — {SYMBOL}")
    axes[0, 1].set_xlabel("Time")
    axes[0, 1].grid(True)

    axes[1, 0].hist(product_df["vol_imbalance"].dropna(), bins=60, color="orange", alpha=0.8, density=True)
    axes[1, 0].axvline(0, color="white", linestyle="--", alpha=0.7)
    axes[1, 0].set_title(f"Volume Imbalance — {SYMBOL}  (positive = more bids)")
    axes[1, 0].set_xlabel("(bid vol − ask vol) / total vol")
    axes[1, 0].grid(True)

    product_df["next_fv_chg"] = product_df["fv"].diff().shift(-1)
    imb_corr = product_df["vol_imbalance"].corr(product_df["next_fv_chg"])
    axes[1, 1].scatter(product_df["vol_imbalance"], product_df["next_fv_chg"], s=1, alpha=0.1, color="white")
    axes[1, 1].set_title(f"Imbalance vs Next-Tick FV Change  (r = {imb_corr:.4f}) — {SYMBOL}")
    axes[1, 1].set_xlabel("Volume Imbalance")
    axes[1, 1].set_ylabel("Next Δ FV")
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.show()

    print(f"Spread — mean: {product_df['spread'].mean():.2f}  std: {product_df['spread'].std():.2f}  "
          f"min: {product_df['spread'].min():.0f}  max: {product_df['spread'].max():.0f}")
    print(f"Vol imbalance vs next return corr: {imb_corr:.4f}")

---
## Market Participant Analysis — Entry, Exit & PnL

In [ ]:
for SYMBOL in SYMBOLS:
    print(f"\n{'='*60}")
    print(f"Market Participant Analysis — {SYMBOL}")
    print(f"{'='*60}")

    product_df_raw = prices_df[prices_df["product"] == SYMBOL]
    product_trades_df = trades_df[trades_df["symbol"] == SYMBOL]

    product_trades = product_trades_df.copy()
    product_trades = product_trades.drop(columns=["currency"])
    product_trades = product_trades.groupby(
        ["timestamp", "price", "day", "buyer", "seller"], as_index=False
    ).agg({"quantity": "sum"})
    product_trades = product_trades.rename(columns={
        "price":    "market order price",
        "quantity": "market order quantity",
    })
    product_df = product_df_raw.merge(
        product_trades[["timestamp", "market order price", "market order quantity", "day", "buyer", "seller"]],
        on=["timestamp", "day"],
        how="left",
    )
    product_df = convert_timestamp(product_df)

    raw_trades = trades_df[trades_df["symbol"] == SYMBOL].copy()
    raw_trades = raw_trades.dropna(subset=["buyer", "seller"])
    raw_trades["t"] = (
        raw_trades["day"].map({2: 0, 3: 1_000_000, 4: 2_000_000})
        + raw_trades["timestamp"]
    )

    if raw_trades.empty:
        print(f"No trades found for {SYMBOL}, skipping participant analysis.")
        continue

    mark_series = product_df.groupby("t")["mid_price"].last()

    def get_mark(t):
        idx = mark_series.index.get_indexer([t], method="nearest")[0]
        return mark_series.iloc[idx]

    vol_min = raw_trades["quantity"].min()
    vol_max = raw_trades["quantity"].max()
    cmap = plt.cm.plasma
    norm = plt.Normalize(vmin=vol_min, vmax=vol_max)

    all_participants = sorted(
        set(raw_trades["buyer"].unique()) | set(raw_trades["seller"].unique())
    )

    def build_equity(trader: str) -> pd.DataFrame:
        buys  = raw_trades[raw_trades["buyer"]  == trader][["t", "price", "quantity"]].copy()
        sells = raw_trades[raw_trades["seller"] == trader][["t", "price", "quantity"]].copy()
        buys["side"]  =  1
        sells["side"] = -1
        df = pd.concat([buys, sells]).sort_values("t").reset_index(drop=True)
        df["cum_pos"]  = (df["side"] * df["quantity"]).cumsum()
        df["cum_cash"] = (-df["side"] * df["price"] * df["quantity"]).cumsum()
        df["mark"]     = df["t"].apply(get_mark)
        df["pnl"]      = df["cum_cash"] + df["cum_pos"] * df["mark"]
        return df

    for trader in all_participants:
        df        = build_equity(trader)
        buys      = df[df["side"] ==  1]
        sells     = df[df["side"] == -1]
        final_pnl = df["pnl"].iloc[-1]
        final_pos = int(df["cum_pos"].iloc[-1])

        fig = plt.figure(figsize=(16, 15))
        gs  = gridspec.GridSpec(4, 1, figure=fig,
                                height_ratios=[3, 1.8, 1.4, 0.9], hspace=0.35)
        ax_price     = fig.add_subplot(gs[0])
        ax_equity    = fig.add_subplot(gs[1], sharex=ax_price)
        ax_inventory = fig.add_subplot(gs[2], sharex=ax_price)
        ax_stats     = fig.add_subplot(gs[3])

        fig.suptitle(
            f"{trader}  —  {SYMBOL}  |  "
            f"Buys: {len(buys)}  Sells: {len(sells)}  |  "
            f"Final PnL: {final_pnl:+.1f}",
            fontsize=13
        )

        ax_price.plot(mark_series.index, mark_series.values,
                      color="white", linewidth=0.7, alpha=0.35, label="Mid price")

        if len(buys):
            ax_price.scatter(buys["t"], buys["price"],
                             c=buys["quantity"], cmap=cmap, norm=norm,
                             s=buys["quantity"] * 16, marker="^",
                             alpha=0.95, zorder=4, label="Buy  ▲")
        if len(sells):
            ax_price.scatter(sells["t"], sells["price"],
                             c=sells["quantity"], cmap=cmap, norm=norm,
                             s=sells["quantity"] * 16, marker="v",
                             alpha=0.95, zorder=4, label="Sell ▽",
                             edgecolors="white", linewidths=0.4)

        for _, row in df.iterrows():
            ax_price.annotate(
                str(int(row["quantity"])),
                xy=(row["t"], row["price"]),
                xytext=(0, 10 if row["side"] == 1 else -14),
                textcoords="offset points",
                ha="center", fontsize=7,
                color=cmap(norm(row["quantity"]))
            )

        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax_price, pad=0.01, shrink=0.85, label="Order size")
        cbar.ax.yaxis.label.set_color("white")
        cbar.ax.tick_params(colors="white")

        ax_price.set_ylabel("Price")
        ax_price.legend(fontsize=8, loc="upper left")
        ax_price.grid(True, alpha=0.3)

        ax_equity.plot(df["t"], df["pnl"], color="gold", linewidth=1.4, label="MTM PnL")
        ax_equity.axhline(0, color="gray", linestyle=":", alpha=0.5)
        ax_equity.fill_between(df["t"], 0, df["pnl"],
                               where=(df["pnl"] >= 0), color="lime", alpha=0.18)
        ax_equity.fill_between(df["t"], 0, df["pnl"],
                               where=(df["pnl"] <  0), color="red",  alpha=0.18)
        ax_equity.scatter(df["t"].iloc[-1], final_pnl,
                          color="gold", s=70, zorder=5,
                          label=f"Final  {final_pnl:+.1f}")
        ax_equity.set_ylabel("PnL")
        ax_equity.legend(fontsize=8)
        ax_equity.grid(True, alpha=0.3)

        ax_inventory.step(df["t"], df["cum_pos"], where="post",
                          color="cyan", linewidth=1.2, label="Inventory")
        ax_inventory.axhline(0, color="gray", linestyle=":", alpha=0.5)
        ax_inventory.fill_between(df["t"], 0, df["cum_pos"],
                                  where=(df["cum_pos"] > 0), color="cyan", alpha=0.15, step="post")
        ax_inventory.fill_between(df["t"], 0, df["cum_pos"],
                                  where=(df["cum_pos"] < 0), color="magenta", alpha=0.15, step="post")
        ax_inventory.scatter(df["t"].iloc[-1], final_pos,
                             color="cyan", s=70, zorder=5,
                             label=f"Final  {final_pos:+d}")
        ax_inventory.set_ylabel("Inventory")
        ax_inventory.set_xlabel("Time")
        ax_inventory.legend(fontsize=8)
        ax_inventory.grid(True, alpha=0.3)

        ax_stats.axis("off")
        stat_keys = ["Trades", "Buys", "Sells", "Total vol",
                     "Avg size", "Max order", "Avg buy px", "Avg sell px",
                     "Final pos", "Final PnL", "Max PnL", "Min PnL"]
        stat_vals = [
            len(df),
            len(buys),
            len(sells),
            int(df["quantity"].sum()),
            f"{df['quantity'].mean():.1f}",
            int(df["quantity"].max()),
            f"{buys['price'].mean():.1f}"  if len(buys)  else "—",
            f"{sells['price'].mean():.1f}" if len(sells) else "—",
            final_pos,
            f"{final_pnl:+.1f}",
            f"{df['pnl'].max():+.1f}",
            f"{df['pnl'].min():+.1f}",
        ]
        tbl = ax_stats.table(
            cellText=[list(map(str, stat_vals))],
            colLabels=stat_keys,
            loc="center",
            cellLoc="center",
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(8.5)
        tbl.scale(1, 2.2)
        for (r, c), cell in tbl.get_celld().items():
            cell.set_facecolor("#111111" if r == 0 else "#1e1e1e")
            cell.set_edgecolor("#3a3a3a")
            cell.set_text_props(color="white")

        plt.show()